# Data Exploration

Use this notebook to explore the raw data before building your dbt models.

**Make sure you've run `uv run python scripts/init_db.py` first.**

## About the Data

The mock data simulates an **online learning platform** where users sign up, enrol in courses, and interact with course content (videos and quizzes).

| Table | Description |
|-------|-------------|
| `raw_users` | User accounts — may contain duplicate records and soft-deleted users |
| `raw_courses` | Course catalog with category, level, and publisher |
| `raw_enrolments` | Which users enrolled in which courses (active, completed, or cancelled) |
| `raw_events` | User activity events: `video_start`, `video_complete`, `quiz_start`, `quiz_submit` |

In [1]:
import duckdb
import pandas as pd

conn = duckdb.connect('../mock_data.duckdb', read_only=True)
print('Connected!')

Connected!


## Raw Tables Overview

In [2]:
# Row counts
for table in ['users', 'courses', 'enrolments', 'events']:
    count = conn.sql(f'SELECT COUNT(*) AS n FROM raw.{table}').fetchone()[0]
    print(f'raw.{table}: {count} rows')

raw.users: 80 rows
raw.courses: 60 rows
raw.enrolments: 90 rows
raw.events: 650 rows


## Users

In [3]:
conn.sql('SELECT * FROM raw.users LIMIT 10').fetchdf()

,id,fullName,email,signupDate,state,isGovEmployee,updatedAt,deleted
0,1,Omar Jones,omar.jones1@example.com,2023-01-04,active,False,2023-02-04 07:08:00,<NA>
1,2,Zoe Jones,zoe.jones2@example.com,2023-03-28,pending,False,2023-04-08 18:27:00,<NA>
2,3,Sam Smith,sam.smith3@example.com,2023-01-12,active,True,2023-03-17 19:01:00,<NA>
3,4,Sky Wilson,sky.wilson4@example.com,2023-04-02,deleted,False,2023-06-10 13:14:00,True
4,5,Pat Robinson,pat.robinson5@example.com,2023-02-05,active,True,2023-05-05 13:21:00,<NA>
5,6,Avery Miller,avery.miller6@example.com,2023-01-28,active,False,2023-02-08 12:06:00,<NA>
6,7,Lee King,lee.king7@example.com,2023-02-14,active,False,2023-02-19 23:29:00,<NA>
7,8,Sky Jones,sky.jones8@example.com,2023-04-29,active,False,2023-07-08 09:53:00,<NA>
8,9,Omar Clark,omar.clark9@example.com,2023-04-24,pending,False,2023-05-18 22:04:00,<NA>
9,10,Sam Lewis,sam.lewis10@example.com,2023-01-30,active,False,2023-02-28 03:24:00,<NA>


In [4]:
# Check for duplicates and deleted users
conn.sql("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_users,
        COUNT(*) FILTER (WHERE deleted = TRUE) AS deleted_users,
        MIN(signupDate) AS earliest_signup,
        MAX(signupDate) AS latest_signup
    FROM raw.users
""").fetchdf()

,total_rows,unique_users,deleted_users,earliest_signup,latest_signup
0,80,70,14,2023-01-01,2023-04-29


## Courses

In [5]:
conn.sql('SELECT * FROM raw.courses LIMIT 10').fetchdf()

,course_id,title,category_name,level,publisher,course_created_at
0,101,Essentials of Procurement 101,Climate,Advanced,LSE,2023-04-04
1,102,Practical Policy 101,Leadership,Advanced,OECD,2023-06-19
2,103,Intro to Climate in Government,Procurement,Intermediate,GovLab,2023-03-25
3,104,Practical Climate in Government,Procurement,Beginner,LSE,2023-04-18
4,105,Mastering Finance 101,Climate,Beginner,UNDP,2023-04-14
5,106,Essentials of Management in Government,Digital,Beginner,GovLab,2023-07-21
6,107,Essentials of HR 101,Leadership,Intermediate,CityU,2023-04-24
7,108,Mastering Data Toolkit,Operations,Advanced,Cambridge,2023-06-18
8,109,Intro to Digital Toolkit,Security,Advanced,OECD,2023-01-24
9,110,Advanced Security in Government,Data,Beginner,Cambridge,2023-01-07


## Enrolments

In [6]:
conn.sql("""
    SELECT 
        COUNT(*) AS total_enrolments,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT course_id) AS unique_courses,
        status, COUNT(*) AS n
    FROM raw.enrolments
    GROUP BY status
""").fetchdf()

,total_enrolments,unique_users,unique_courses,status,n
0,30,22,25,cancelled,30
1,28,23,25,active,28
2,32,30,26,completed,32


## Events

In [7]:
conn.sql('SELECT * FROM raw.events LIMIT 10').fetchdf()

,id,user_id,course_id,event_type,event_timestamp,session_id,metadata
0,9001,55,113,video_start,2023-03-03 09:02:42,ikcidk,None
1,9002,53,142,video_start,2023-03-03 12:46:15,fn9xuy,None
2,9003,61,104,video_complete,2023-03-03 12:52:02,2oc6uz,None
3,9004,6,149,video_start,2023-03-05 15:10:50,yvnvqj,None
4,9005,1,144,video_start,2023-03-05 12:43:56,tjxepq,None
5,9006,70,117,video_complete,2023-03-05 12:40:19,4shnf8,None
6,9007,19,116,video_start,2023-03-06 14:05:05,039tex,None
7,9008,10,117,quiz_submit,2023-03-06 19:13:59,t0hl9x,quiz:fail
8,9009,43,108,video_complete,2023-03-06 14:37:35,ihcwi6,None
9,9010,7,133,video_complete,2023-03-06 17:17:42,rt05ui,None


In [8]:
# Event type distribution
conn.sql("""
    SELECT event_type, COUNT(*) AS n
    FROM raw.events
    GROUP BY event_type
    ORDER BY n DESC
""").fetchdf()

,event_type,n
0,quiz_submit,178
1,video_start,171
2,video_complete,161
3,quiz_start,140


In [9]:
# Date range and coverage
conn.sql("""
    SELECT 
        COUNT(*) AS total_events,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT course_id) AS unique_courses,
        MIN(event_timestamp::DATE) AS first_event,
        MAX(event_timestamp::DATE) AS last_event,
        COUNT(DISTINCT event_timestamp::DATE) AS active_days
    FROM raw.events
""").fetchdf()

,total_events,unique_users,unique_courses,first_event,last_event,active_days
0,650,57,60,2023-03-03,2023-05-29,87


---

## Verify Your Results

Run the cells below **after** you've built and run your dbt models (`uv run dbt run --profiles-dir .`).

Your dbt models are materialized in the same DuckDB database under the `analytics` schema, so you can query them directly here.

In [10]:
import duckdb
import pandas as pd

conn = duckdb.connect('../mock_data.duckdb', read_only=True)
print('Connected!')

Connected!


In [11]:
# Load raw CSVs
users = pd.read_csv('../mock_data/raw_users.csv')
events = pd.read_csv('../mock_data/raw_events.csv', parse_dates=['event_timestamp'])

# Deduplicate users: keep latest record per id, exclude deleted
users = users.sort_values('updatedAt', ascending=False).drop_duplicates(subset='id')
valid_users = users[users['deleted'] != True]

# Filter events to valid (non-deleted) users only
events = events[events['user_id'].isin(valid_users['id'])].copy()
events['event_date'] = events['event_timestamp'].dt.date

print(f"Valid users: {len(valid_users)}")
print(f"Events (non-deleted users): {len(events)}")

Valid users: 57
Events (non-deleted users): 650


### Task 1 — DAU spot check

Expected DAU = distinct non-deleted users with any event on that date. Compare this against your model's `dau` column.

In [12]:
# DAU from raw data: distinct users per date
# Reindex to full date range so days with no events show DAU = 0
dau_raw = (
    events
    .groupby('event_date')['user_id']
    .nunique()
)
all_dates = pd.date_range(events['event_date'].min(), events['event_date'].max(), freq='D')
expected_dau = (
    dau_raw
    .reindex(all_dates.date, fill_value=0)
    .reset_index()
    .rename(columns={'index': 'event_date', 'user_id': 'expected_dau'})
)
print(f"{len(expected_dau)} rows (one per calendar date, no gaps)")
expected_dau

88 rows (one per calendar date, no gaps)


,event_date,expected_dau
0,2023-03-03,3
1,2023-03-04,0
2,2023-03-05,3
3,2023-03-06,7
4,2023-03-07,3
...,...,...
83,2023-05-25,10
84,2023-05-26,9
85,2023-05-27,5
86,2023-05-28,6


In [13]:
# Compare against your dbt model
# Replace 'YOUR_MODEL_NAME' with your actual metrics model name
try:
    model_rau = conn.sql("""
        SELECT window_end_date, dau, wau, mau
        FROM analytics.YOUR_MODEL_NAME
        ORDER BY window_end_date
    """).fetchdf()
    model_rau['window_end_date'] = pd.to_datetime(model_rau['window_end_date']).dt.date

    comparison = model_rau.merge(expected_dau, left_on='window_end_date', right_on='event_date', how='left')
    comparison['expected_dau'] = comparison['expected_dau'].fillna(0).astype(int)
    comparison['dau_match'] = comparison['dau'] == comparison['expected_dau']

    mismatches = comparison[~comparison['dau_match']]
    if len(mismatches) == 0:
        print(f"DAU check PASSED — all {len(comparison)} rows match")
    else:
        print(f"DAU check FAILED — {len(mismatches)} mismatches:")
        print(mismatches[['window_end_date', 'dau', 'expected_dau']].to_string(index=False))
except Exception as e:
    print(f"Could not query model: {e}")
    print("Update YOUR_MODEL_NAME above with the name of your metrics table.")

Could not query model: Catalog Error: Table with name YOUR_MODEL_NAME does not exist!
Did you mean "base_raw__users"?
Update YOUR_MODEL_NAME above with the name of your metrics table.


### Task 1 — WAU spot check

Pick a specific date and manually count distinct users in the 7-day window to verify your WAU.

In [14]:
# WAU spot check: pick a date, count distinct users in trailing 7 days
from datetime import date, timedelta

check_date = date(2023, 4, 5)  # change this to spot-check other dates
window_start = check_date - timedelta(days=6)

In [15]:
users_in_window = events[
    (events['event_date'] >= window_start) & (events['event_date'] <= check_date)
]['user_id'].nunique()

print(f"WAU for {check_date} (window {window_start} to {check_date}):")
print(f"  Expected distinct users: {users_in_window}")

# Show who they are
user_list = sorted(events[
    (events['event_date'] >= window_start) & (events['event_date'] <= check_date)
]['user_id'].unique().tolist())
print(f"  User IDs: {user_list}")

WAU for 2023-04-05 (window 2023-03-30 to 2023-04-05):
  Expected distinct users: 28
  User IDs: [1, 2, 3, 5, 6, 10, 11, 14, 15, 18, 23, 26, 29, 30, 32, 36, 38, 40, 43, 45, 47, 57, 61, 62, 63, 64, 66, 70]


### Task 1 — MAU spot check

Pick a specific date and manually count distinct users in the 28-day window to verify your MAU.

In [16]:
# MAU spot check: pick a date, count distinct users in trailing 28 days
check_date = date(2023, 4, 5)  # change this to spot-check other dates
window_start = check_date - timedelta(days=27)

In [17]:
users_in_window = events[
    (events['event_date'] >= window_start) & (events['event_date'] <= check_date)
]['user_id'].nunique()

print(f"MAU for {check_date} (window {window_start} to {check_date}):")
print(f"  Expected distinct users: {users_in_window}")

user_list = sorted(events[
    (events['event_date'] >= window_start) & (events['event_date'] <= check_date)
]['user_id'].unique().tolist())
print(f"  User IDs: {user_list}")

MAU for 2023-04-05 (window 2023-03-09 to 2023-04-05):
  Expected distinct users: 41
  User IDs: [1, 2, 3, 5, 6, 7, 10, 11, 14, 15, 18, 19, 21, 23, 26, 27, 28, 29, 30, 32, 33, 36, 37, 38, 40, 43, 45, 46, 47, 49, 52, 53, 54, 55, 57, 61, 62, 63, 64, 66, 70]


### Task 1 Bonus — RAU by event type spot check

If you segmented RAU by event type, verify by filtering raw events to a single type.

In [18]:
# Spot check RAU for a specific event type
check_date = date(2023, 4, 5)  # change this to spot-check other dates
check_event_type = 'quiz_submit'  # change to: video_start, video_complete, quiz_start, quiz_submit

type_events = events[events['event_type'] == check_event_type]

dau = type_events[type_events['event_date'] == check_date]['user_id'].nunique()

wau_start = check_date - timedelta(days=6)
wau = type_events[
    (type_events['event_date'] >= wau_start) & (type_events['event_date'] <= check_date)
]['user_id'].nunique()

mau_start = check_date - timedelta(days=27)
mau = type_events[
    (type_events['event_date'] >= mau_start) & (type_events['event_date'] <= check_date)
]['user_id'].nunique()

print(f"RAU for '{check_event_type}' on {check_date}:")
print(f"  DAU: {dau}")
print(f"  WAU: {wau} (window {wau_start} to {check_date})")
print(f"  MAU: {mau} (window {mau_start} to {check_date})")

RAU for 'quiz_submit' on 2023-04-05:
  DAU: 3
  WAU: 9 (window 2023-03-30 to 2023-04-05)
  MAU: 23 (window 2023-03-09 to 2023-04-05)


### Task 2 — Cohort retention spot check

Pick a cohort week and verify cohort_size and active_users for a given period.

In [19]:
# Cohort spot check from raw CSVs
valid_users_df = valid_users.copy()
valid_users_df['signupDate'] = pd.to_datetime(valid_users_df['signupDate'])
valid_users_df['cohort_week'] = valid_users_df['signupDate'].dt.to_period('W-SUN').apply(lambda p: p.start_time.date())

# Pick a cohort to inspect
check_cohort = date(2023, 3, 20)  # change this to spot-check other cohorts
cohort_users = valid_users_df[valid_users_df['cohort_week'] == check_cohort]
cohort_size = len(cohort_users)
cohort_user_ids = set(cohort_users['id'])

print(f"Cohort {check_cohort}: {cohort_size} users — IDs: {sorted(cohort_user_ids)}")

Cohort 2023-03-20: 3 users — IDs: [11, 32, 45]


In [20]:
# Check activity per week for this cohort
cohort_events = events[events['user_id'].isin(cohort_user_ids)].copy()
cohort_events['activity_week'] = pd.to_datetime(cohort_events['event_date']).dt.to_period('W-SUN').apply(lambda p: p.start_time.date())
cohort_events['periods_since'] = cohort_events['activity_week'].apply(
    lambda w: (w - check_cohort).days // 7
)

retention = (
    cohort_events[cohort_events['periods_since'] >= 0]
    .groupby('periods_since')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'active_users'})
)
retention['cohort_size'] = cohort_size
retention['retention_rate'] = (retention['active_users'] / cohort_size).round(2)
retention

,periods_since,active_users,cohort_size,retention_rate
0,0,1,3,0.33
1,1,3,3,1.00
2,2,1,3,0.33
3,3,3,3,1.00
4,4,3,3,1.00
5,5,3,3,1.00
6,6,3,3,1.00
7,7,3,3,1.00
8,8,2,3,0.67
9,9,3,3,1.00


## Terminate duckdb connection

In [21]:
conn.close()